# Notebook 09c: the possible futures of Idea 9, and the external reach arm

## STATUS: SIMULATION AND SCAFFOLDING ONLY. There is no empirical result in this notebook.

Nothing here measures the encoder, the cohort, or any checkpoint. Every lane R-squared, p-value, variance
fraction, and mirror slope in sections 2 through 4 is a number a human CHOSE, in order to see what the
decision rule would do with it. Every correlation and slope in section 5 comes from a synthetic fixture
whose answer was PLANTED by construction. Read this notebook as a pre-registration aid and a code
scaffold. Do not cite any number in it as a finding.

## Orientation: what this notebook is for

`nb_09a` runs arm 1, the zero-retrain antisymmetric readout on the frozen `ea59fea0` checkpoint, and
returns one verdict. `nb_09b` is arm 2's smoke-scale scaffold, and arm 2's real path is the
`new_nb_09_00` through `new_nb_09_03` series. This notebook does two things neither of those can.

1. **A possible-futures simulator.** Before real numbers land, it is worth knowing exactly what each
   outcome would LOOK like and precisely which claim it would license. So we write down the canonical
   futures of the Idea 9 endpoint, score each one against the SAME hardened gates `nb_09a` uses, and draw
   the shape each would make on the two decisive plots. The value of doing this in advance is that the
   decision rule cannot then be tuned to a result that has already been seen.
2. **An external multi-view reach scaffold.** The claim on this cohort is internal-validity only, at 18
   source videos with a transductive encoder. The reflection-equivariance PROPERTY, however, has a
   NON-clinical reach test on public multi-view pose cohorts (CASIA-B, OU-MVLP-Pose). Section 5 lays down
   a runnable, honestly stubbed loader interface for that arm. The real loaders are marked TODO and are
   not wired.

## The one thing this notebook got wrong, stated up front

Its future **F3 predicted an INFORMATIVE NULL for arm 1**. The **ACTUAL arm 1 verdict was
`ARTIFACT (side-agnostic nuisance control fired)`**. That is a genuine and interesting miss, and it is left
in place rather than quietly rewritten to match the result. Section 2 explains what the rehearsal failed to
anticipate: that a control which is mathematically BLIND to left and right would OUTSCORE the treatment
built specifically to read left from right. Section 3 draws the lesson, which is about control ladders
needing to be richer than the set of outcomes you expect.

## What the reader is assumed to have, and what the gates are

Assumed: a reading of `nb_09a`, at least through its verdict section. The gates below are copied to match
`nb_09a` exactly, so a future cannot be scored on a moved goalpost:

- beat the binding bar `max(D_standard, C_floor)` by at least 0.05 R-squared;
- beat the untrained-encoder floor `C` by at least 0.05;
- beat the capacity-matched control `Ac` (the same head, with the symmetric left-plus-right path added) by
  at least 0.05;
- clear a source-label permutation null at p < 0.05;
- keep the side-blind nuisance control `E` below 0.05 in ABSOLUTE value;
- clear the y-quality gate, that is the target's between-source variance fraction reaches 0.30.

The exact wiring-swap slope of -1 is a separate by-construction check (`nb_09a` section 5) and is
therefore NOT a future: it is true for every possible encoder and so distinguishes nothing.

## Standing caveats

See `notes/ideas-claude/_shared_facts.md` for the shared numbers. Folder labels are dataset annotations,
not clinical diagnoses. All results on this cohort are transductive, the source video is the independent
unit of evidence, and nothing in this notebook or the ones it discusses is clinical validation.


## 0. The pre-registered gates (frozen, matched to nb_09a)

**Step 1 of 7.**

*What we are about to do.* Define the five gate constants and the mirror band, and print them, so the
decision rule is visible in the output rather than buried in code.

*Why they are defined here and never touched again.* These constants ARE the decision rule for the primary
endpoint, and they match `nb_09a`'s verdict cell. Freezing them in the first code cell means no future can
be scored on a goalpost that moved after the fact. The constants, each defined before it is used:

- `FLOOR_MARGIN = 0.05`: the R-squared margin by which lane A_prime must beat the binding bar
  `max(D, C)`, the floor `C`, AND the capacity-matched control `Ac`.
- `PERM_ALPHA = 0.05`: the significance level A_prime's source-label permutation null must clear.
- `NUISANCE_ABS = 0.05`: the ABSOLUTE bound on the side-blind nuisance lane E.
- `Y_BETWEEN_MIN = 0.30`: the minimum between-source fraction of the target's variance required before a
  held-out-source R-squared is interpretable at all.
- `MIRROR_BAND = (-1.25, -0.8)`: the band a MEASURED anatomical-mirror slope must fall inside to count as
  flipping. This is measured through the encoder and is emphatically not the wiring -1.

*Which three of these are new relative to Idea 5, and what escape route each closes.* This is the
load-bearing hardening, so take them one at a time.

1. **The binding bar is the LARGER of the standard-encoder comparator D and the untrained floor C.** The
   escape route it closes: beating only the weaker of the two and presenting that as a win.
2. **A source-label permutation null replaces Idea 5's fixed sign-consistency threshold.** The escape
   route it closes: assuming a bar like "correct on 75 percent of sources" is meaningful at n=18 sources
   with per-condition counts as low as 1, where its sampling distribution is wide and lumpy.
3. **A capacity-matched control Ac must be beaten by the same 0.05 margin.** The escape route it closes: a
   win that actually came from the head's nonlinearity, width, or pair information rather than from the
   antisymmetry CONSTRAINT. Ac holds all of those identical and differs only by adding the
   mirror-invariant symmetric path.

*What to look at in the output.* The resolved `IDEA9_DIR`, then the six printed lines of the decision rule.
Nothing is measured; these are constants being echoed.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_SEED = 42


def idea9_dir():
    '''Resolve the 09 proposal folder robustly regardless of the kernel cwd.'''
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / "notes" / "ideas-claude" / "09-reflection-equivariant-symmetry-axis"
        if cand.exists():
            return cand
        cand = base / "09-reflection-equivariant-symmetry-axis"
        if cand.exists():
            return cand
    return Path.cwd()


IDEA9_DIR = idea9_dir()
print(f"IDEA9_DIR: {IDEA9_DIR}")

# ---- pre-registered gates (match nb_09a section 10) ----
FLOOR_MARGIN = 0.05          # A' must beat the binding bar max(D,C), the floor C, AND Ac by >= this R2.
PERM_ALPHA = 0.05            # A' must clear its source-label permutation null at p < this.
NUISANCE_ABS = 0.05          # |E_pooled R2| must be < this ABSOLUTELY (no OR-clause).
Y_BETWEEN_MIN = 0.30         # y between-source variance fraction must be >= this to trust R2.
MIRROR_BAND = (-1.25, -0.8)  # measured anatomical-mirror slope inside this band counts as "flips".

print("Pre-registered decision rule (matches nb_09a):")
print(f"  beat binding bar max(D,C) by     >= {FLOOR_MARGIN} R2")
print(f"  beat untrained floor C by        >= {FLOOR_MARGIN} R2")
print(f"  beat capacity-matched Ac by      >= {FLOOR_MARGIN} R2  (attribution to antisymmetry)")
print(f"  A' permutation null              p  < {PERM_ALPHA}")
print(f"  |E_pooled nuisance R2|           <  {NUISANCE_ABS}  (absolute, anatomically invariant)")
print(f"  y between-source variance frac   >= {Y_BETWEEN_MIN}")
print(f"  anatomical-mirror slope band      = {MIRROR_BAND}  (measured, NOT the wiring -1)")

IDEA9_DIR: /Users/pmui/dev/alexpose/experiments/sjepa/gavd6-pm/notes/ideas-claude/09-reflection-equivariant-symmetry-axis
Pre-registered decision rule (matches nb_09a):
  beat binding bar max(D,C) by     >= 0.05 R2
  beat untrained floor C by        >= 0.05 R2
  beat capacity-matched Ac by      >= 0.05 R2  (attribution to antisymmetry)
  A' permutation null              p  < 0.05
  |E_pooled nuisance R2|           <  0.05  (absolute, anatomically invariant)
  y between-source variance frac   >= 0.3
  anatomical-mirror slope band      = (-1.25, -0.8)  (measured, NOT the wiring -1)


## 1. One scoring function, shared by every future

**Step 2 of 7.**

*What we are about to do.* Write `score_future`, the single function that turns a set of lane values into
the exact verdict language Idea 9 pre-registers, then self-check it on four hand-built cases.

*Why one shared function matters.* If each future were scored by its own ad hoc reasoning, the "decision
rule" would not be a rule. Routing every future through one function guarantees that the same inputs
always produce the same verdict, and that the function can be compared line by line against `nb_09a`'s
verdict cell.

*What it takes as arguments, all of them simulated inputs in this notebook.* The five gated lane
R-squared values (A_prime the antisymmetric head, Ac the capacity-matched control, C the untrained floor,
D the standard `ea59fea0` comparator, E the side-blind nuisance control), the A_prime permutation p-value,
the C permutation p-value, the target's between-source variance fraction, and the MEASURED
anatomical-mirror slope.

*The rule, stated in the order the code checks it.* The primary verdict is a positive ONLY if every gate
passes together. Otherwise the code reports the FIRST applicable failure mode, and the order encodes a
priority:

1. nuisance control fired, so the signed claim is withdrawn as an ARTIFACT;
2. the y-quality gate failed, so the result is UNINTERPRETABLE;
3. the gates passed except the capacity-matched control, so the win is NOT ATTRIBUTABLE to antisymmetry;
4. everything passed, so ANTISYMMETRY BEATS BINDING BAR AND CAPACITY-MATCHED CONTROL;
5. otherwise, an INFORMATIVE NULL.

*Why "artifact" is checked before everything else.* Because it is not a weaker or stronger result, it is a
different kind of statement. If a side-blind control recovers the axis, then the lane is not admissible
evidence about sides at all, and there is no point ranking a number that cannot speak to the question.

*What the C permutation is, and what it is NOT.* It is REPORTED as floor characterisation only. A
significant C means the untrained floor's random features preserve genuine laterality, which just raises
the binding bar; it is not evidence of source-identity leakage and it is not a claim-withdrawing gate.
This matches `nb_09a` exactly. Lane B, the raw-coordinate ceiling, is likewise descriptive only, because
it is near-circular.

*What to look at in the output.* One line,
`scoring self-check OK (clean positive passes; nuisance-fire and not-attributable traps withhold the
claim; a significant C-null does NOT withdraw a clean positive).` The four assertions above it are the
substance: a clean positive passes; a clean positive is unaffected by a significant C null; a strong
A_prime with the nuisance control firing is withheld; and a strong A_prime that fails to beat Ac is
reported as not attributable.

*What we may conclude.* That the decision rule is total and self-consistent on hand-built inputs. Nothing
about the encoder: every value fed to it here was chosen by hand.


In [2]:
def score_future(a_r2, ac_r2, c_r2, d_r2, e_r2, a_perm_p, c_perm_p, y_between_frac, mirror_slope):
    binding_bar = max(d_r2, c_r2)
    beats_binding = (a_r2 - binding_bar) >= FLOOR_MARGIN
    beats_floor = (a_r2 - c_r2) >= FLOOR_MARGIN
    # Attribution gate: A' must beat the capacity-matched control Ac (same head, adds the symmetric
    # left-plus-right path). A gap here is attributable to the antisymmetry CONSTRAINT, not the head's
    # nonlinearity, width, or pair information (which Ac holds identical).
    beats_capacity_matched = (a_r2 - ac_r2) >= FLOOR_MARGIN
    perm_ok = a_perm_p < PERM_ALPHA
    nuisance_ok = abs(e_r2) < NUISANCE_ABS
    y_ok = y_between_frac >= Y_BETWEEN_MIN
    # C's own permutation is REPORTED but is NOT a gate: a random floor that preserves genuine laterality
    # is a STRONG floor (it raises the binding bar), not a broken null. This matches nb_09a exactly.
    c_null_significant = c_perm_p < PERM_ALPHA
    primary_positive = bool(beats_binding and beats_floor and beats_capacity_matched and perm_ok
                            and nuisance_ok and y_ok)
    flips = bool(MIRROR_BAND[0] <= mirror_slope <= MIRROR_BAND[1])
    if not nuisance_ok:
        primary = "ARTIFACT (nuisance control fired; signed claim withdrawn)"
    elif not y_ok:
        primary = "UNINTERPRETABLE (y is noise-dominated at the source level)"
    elif (beats_binding and beats_floor and perm_ok) and not beats_capacity_matched:
        primary = "NOT ATTRIBUTABLE TO ANTISYMMETRY (does not beat the capacity-matched control)"
    elif primary_positive:
        primary = "ANTISYMMETRY BEATS BINDING BAR AND CAPACITY-MATCHED CONTROL"
    else:
        primary = "INFORMATIVE NULL (constrained head does not beat max(D,C) above the floor)"
    mirror = ("FLIPS (measured slope antisymmetric through the encoder)" if flips
              else "DOES NOT FLIP (measured slope not antisymmetric)")
    return {
        "beats_binding_bar": beats_binding, "beats_floor": beats_floor,
        "beats_capacity_matched": beats_capacity_matched,
        "perm_ok": perm_ok, "nuisance_ok": nuisance_ok, "y_ok": y_ok,
        "c_null_significant": c_null_significant, "binding_bar": float(binding_bar),
        "primary_positive": primary_positive, "flips": flips,
        "primary_verdict": primary, "mirror_verdict": mirror,
    }


# sanity: a clean positive (A'=0.42 beats binding bar max(D=-0.02,C=0.05)=0.05 AND Ac=0.10, perm passes).
_demo = score_future(0.42, 0.10, 0.05, -0.02, 0.01, 0.004, 0.60, 0.55, -1.0)
assert _demo["primary_positive"] and _demo["flips"] and _demo["beats_capacity_matched"], _demo
# sanity: a clean positive is UNAFFECTED by a significant C-null (C is descriptive, not a gate).
_demo_strongC = score_future(0.42, 0.10, 0.05, -0.02, 0.01, 0.004, 0.01, 0.55, -1.0)
assert _demo_strongC["primary_positive"] and _demo_strongC["c_null_significant"], _demo_strongC
# sanity: nuisance firing withdraws the claim even with a strong A'.
_art = score_future(0.47, 0.10, 0.06, -0.02, 0.46, 0.004, 0.60, 0.55, -0.9)
assert (not _art["primary_positive"]) and (not _art["nuisance_ok"]), _art
# sanity: a strong A' that does NOT beat the capacity-matched control Ac is not attributable (trap F5).
_notattr = score_future(0.30, 0.28, 0.05, -0.02, 0.01, 0.02, 0.60, 0.50, -0.7)
assert (not _notattr["primary_positive"]) and (not _notattr["beats_capacity_matched"]), _notattr
print("scoring self-check OK (clean positive passes; nuisance-fire and not-attributable traps withhold the "
      "claim; a significant C-null does NOT withdraw a clean positive).")

scoring self-check OK (clean positive passes; nuisance-fire and not-attributable traps withhold the claim; a significant C-null does NOT withdraw a clean positive).


## 2. The canonical possible futures

**Step 3 of 7. Every number in this section is SIMULATED. They are illustrative expected shapes chosen by
hand, not measurements.**

*What we are about to do.* Name five plausible states of the world, give each the lane values it would
produce, and run all five through `score_future`.

*Why we do this before the result rather than after.* So that the mapping from outcome to licensed claim
is fixed in advance. A decision rule written after seeing the data is not a decision rule.

### The five futures, with what each would license

- **F1, head beats bar (positive).** The antisymmetry-constrained head decodes the signed axis above the
  binding bar `max(D, C)`, beats the capacity-matched control `Ac`, clears its permutation null, and the
  measured anatomical mirror flips it. For arm 1 this would say the CONSTRAINT extracts a signed axis the
  unconstrained probe missed; for arm 2 it would say the equivariance loss BUILT one. The strong result.
- **F2, decodable but non-flipping.** The head beats the bar and `Ac` and clears its null, but the
  measured mirror slope falls outside the flip band. Licenses the decodability claim, withholds the
  measured-equivariance claim. The exact wiring -1 still holds by construction and is a separate matter.
- **F3, informative null.** The constrained head does not clear the binding bar above the floor. Overturns
  the hope that antisymmetry-by-construction ALONE rescues a signed axis on the frozen encoder: a clean
  publishable negative. This was the rehearsal's expected arm 1 outcome, and section 3 explains why it was
  wrong.
- **F4, artifact.** The head looks strong BUT the side-blind nuisance control E also "recovers" the axis,
  which is impossible for a genuinely signed quantity. The signed claim is withdrawn.
- **F5, not-attributable trap.** The head beats the binding bar and clears its null, but does not beat
  `Ac` by the margin, so the win rides on the head's generic capacity rather than on the antisymmetry
  constraint. Reported as not attributable rather than as a constraint win.

### IMPORTANT: F3's anchor values are SUPERSEDED inputs

F3 was seeded from what was then believed to be the current Idea 5 result: A learned **-0.187**, C floor
**+0.147**, D pooled **-0.014**, mirror slope **-0.34**. Those values came from a **superseded Idea 5
bundle carrying the `d0acc262` fingerprint**, which is a stale leftover from an earlier lineage. They are
NOT the current Idea 5 result and must not be quoted as such anywhere.

The authoritative Idea 5 result, on the same `ea59fea0` checkpoint that arm 1 uses, is:

| Idea 5 lane | Superseded `d0acc262` value seeded into F3 | Authoritative `ea59fea0` value |
|---|---|---|
| A_learned | -0.187 | **-0.602** |
| C_floor | +0.147 | **-0.156** |
| D_pooled | -0.014 | **-0.131** |
| anatomical mirror slope | -0.34 (the bundle records -0.343) | **-0.741** |

The code cell below still carries the superseded numbers, because they are the simulated inputs that were
actually used when these futures were scored and rewriting them would misrepresent what was run. Read F3
as a simulated future whose ANCHOR was drawn from a stale bundle. Notably, its verdict does not depend on
the correction: under the authoritative values the constrained head still fails to clear the binding bar,
so F3 would still have been scored an informative null.

One knock-on effect worth naming: the superseded C floor was POSITIVE at +0.147, which is why F3 carries a
significant C permutation p of 0.01 and the commentary about a strong untrained floor. On the
authoritative numbers the floor is negative, and in the real arm 1 run C's permutation p came out at 0.388,
that is not significant at all.

*What to look at in the output.* The five-row futures table. Read the `primary` column against the input
columns to see the rule working: F1 and F2 both reach the positive verdict and differ only in the mirror
column; F3 lands on the informative null; F4 lands on the artifact because `nuisance_ok` is False; and F5
lands on not-attributable because `beats_capacity_matched` is False.

*What we may NOT conclude.* Nothing empirical whatsoever. Every input in this table was chosen by hand to
illustrate a decision boundary.


In [3]:
FUTURES = [
    {"key": "F1_head_beats_bar", "title": "F1 head beats bar (positive)",
     "a_r2": 0.42, "ac_r2": 0.10, "c_r2": 0.05, "d_r2": -0.02, "e_r2": 0.01,
     "a_perm_p": 0.004, "c_perm_p": 0.60, "y_between_frac": 0.55, "mirror_slope": -1.02},
    {"key": "F2_decodable_non_flip", "title": "F2 decodable but non-flipping",
     "a_r2": 0.40, "ac_r2": 0.09, "c_r2": 0.05, "d_r2": -0.02, "e_r2": 0.01,
     "a_perm_p": 0.01, "c_perm_p": 0.55, "y_between_frac": 0.52, "mirror_slope": -0.30},
    {"key": "F3_informative_null", "title": "F3 informative null (Idea-05 anchor)",
     "a_r2": -0.10, "ac_r2": -0.05, "c_r2": 0.147, "d_r2": -0.187, "e_r2": -0.014,
     "a_perm_p": 0.55, "c_perm_p": 0.01, "y_between_frac": 0.48, "mirror_slope": -0.34},
    {"key": "F4_artifact", "title": "F4 artifact (nuisance control fires)",
     "a_r2": 0.47, "ac_r2": 0.11, "c_r2": 0.06, "d_r2": -0.02, "e_r2": 0.46,
     "a_perm_p": 0.004, "c_perm_p": 0.55, "y_between_frac": 0.52, "mirror_slope": -0.9},
    {"key": "F5_not_attributable", "title": "F5 not attributable (Ac not beaten)",
     "a_r2": 0.30, "ac_r2": 0.28, "c_r2": 0.05, "d_r2": -0.02, "e_r2": 0.01,
     "a_perm_p": 0.02, "c_perm_p": 0.60, "y_between_frac": 0.50, "mirror_slope": -0.7},
]

rows = []
for f in FUTURES:
    v = score_future(f["a_r2"], f["ac_r2"], f["c_r2"], f["d_r2"], f["e_r2"],
                     f["a_perm_p"], f["c_perm_p"], f["y_between_frac"], f["mirror_slope"])
    rows.append({
        "future": f["title"], "A'_r2": f["a_r2"], "Ac_r2": f["ac_r2"], "C_r2": f["c_r2"],
        "D_r2": f["d_r2"], "E_r2": f["e_r2"], "binding_bar": v["binding_bar"],
        "A'_perm_p": f["a_perm_p"], "C_perm_p": f["c_perm_p"],
        "y_between": f["y_between_frac"], "mirror_slope": f["mirror_slope"],
        "primary": v["primary_verdict"], "mirror": v["mirror_verdict"],
        "nuisance_ok": v["nuisance_ok"], "beats_capacity_matched": v["beats_capacity_matched"],
        "c_null_significant": v["c_null_significant"],
    })
futures_table = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 60)
print(futures_table.to_string(index=False))

                              future  A'_r2  Ac_r2  C_r2   D_r2   E_r2  binding_bar  A'_perm_p  C_perm_p  y_between  mirror_slope                                                                       primary                                                   mirror  nuisance_ok  beats_capacity_matched  c_null_significant
        F1 head beats bar (positive)   0.42   0.10 0.050 -0.020  0.010        0.050      0.004      0.60       0.55         -1.02                   ANTISYMMETRY BEATS BINDING BAR AND CAPACITY-MATCHED CONTROL FLIPS (measured slope antisymmetric through the encoder)         True                    True               False
       F2 decodable but non-flipping   0.40   0.09 0.050 -0.020  0.010        0.050      0.010      0.55       0.52         -0.30                   ANTISYMMETRY BEATS BINDING BAR AND CAPACITY-MATCHED CONTROL         DOES NOT FLIP (measured slope not antisymmetric)         True                    True               False
F3 informative null (Idea-05 ancho

## 3. The decision table: future to licensed claim

**Step 4 of 7. Still entirely simulated.**

*What we are about to do.* Print, for each of the five simulated futures, the verdict AND the exact claim
language that verdict licenses, together with what it withholds.

*Why this table is the single artifact a reviewer should read first.* A verdict string is not a claim. The
distance between "the gates passed" and "here is what may be written in a paper" is where overclaiming
happens, so that mapping is written down explicitly and in advance.

*The two traps, which are the reason the ladder has six lanes rather than three.* In both F4 (the nuisance
control fires) and F5 (the capacity-matched control is not beaten), lane A_prime looks STRONG and is
nonetheless not enough on its own. Any ladder that reported only the treatment would have called both of
them successes.

### The rehearsal's genuine miss, and the lesson it teaches

*The prediction.* F3 was named as the expected arm 1 outcome, and its licensed claim reads that
antisymmetry-by-construction alone does not lift the frozen encoder above the binding bar: an informative
null.

*The actual result.* Arm 1's verdict in `nb_09a` was **`ARTIFACT (side-agnostic nuisance control fired)`**,
which in this notebook's own taxonomy is F4, not F3. The side-blind lane E scored **-0.066** against the
antisymmetric treatment A_prime at **-0.206**, so E outscored the treatment by **0.140**.

*What the rehearsal failed to anticipate, stated precisely.* Not that arm 1 would fail. It predicted
failure correctly. What it did not anticipate is the MECHANISM of the failure: that a control which is
mathematically blind to left and right would score HIGHER than the lane built specifically to read left
from right. F4 was written as the scenario where a strong-looking A_prime is undercut by a nuisance lane.
The real outcome was stranger, a WEAK A_prime undercut by a nuisance lane that was less weak. The
rehearsal treated F3 and F4 as far apart, with F4 requiring `A_prime` around +0.47; in the event, the
artifact verdict was reached with `A_prime` at -0.206.

*The lesson, which is the point of keeping the miss visible.* Control ladders must be richer than the set
of outcomes you expect. Had the ladder contained only the treatment, the floor, and the raw ceiling, arm 1
would have been written up as an informative null exactly as rehearsed, and the null would have been
WRONG, not because the direction was wrong but because the lane was never admissible evidence about sides
in the first place. It was lane E, a control included for completeness against a scenario nobody expected
to fire this way, that revealed it. The generalisation: include a control for each escape route you can
NAME, not only for each outcome you can PREDICT.

*Why the prediction is not being rewritten to match.* Because a possible-futures rehearsal that is edited
after the fact to agree with the result has no evidential value at all, and because the disagreement is
itself the finding of this section.

*What to look at in the output.* Five blocks, one per simulated future, each with its primary verdict,
mirror verdict, control flags, and licensed-claim text. The F1 claim names the authoritative `ea59fea0`
standard comparator.

*What we may NOT conclude.* Nothing empirical. These are five hand-built scenarios and their pre-committed
claim language.


In [4]:
CLAIMS = {
    "F1 head beats bar (positive)": (
        "LICENSED: the antisymmetry-constrained head decodes the signed axis above the binding bar "
        "max(standard-ea59fea0, untrained-floor), beats the capacity-matched control Ac, AND clears its "
        "permutation null, and the MEASURED anatomical mirror flips it. Arm 1: the constraint extracts a "
        "signed axis the unconstrained probe missed. Arm 2: the equivariance loss built one. (The exact "
        "wiring -1 is a separate by-construction check, always true.)"),
    "F2 decodable but non-flipping": (
        "LICENSED: signed axis is decodable above the binding bar and the capacity-matched control. "
        "WITHHELD: the MEASURED anatomical equivariance; the encoding decodes side but the through-encoder "
        "mirror slope is not in the flip band. The exact wiring-swap -1 still holds by construction."),
    "F3 informative null (Idea-05 anchor)": (
        "LICENSED (negative): antisymmetry-by-construction alone does NOT lift the frozen encoder above the "
        "binding bar. Matches Idea 05's measured result and is the expected Arm-1 outcome; a clean "
        "publishable negative for the representation audit. The significant C-permutation here is REPORTED "
        "as floor characterization (the untrained floor is strong, which just raises the binding bar) and "
        "does not change the verdict. Motivates Arm 2 (retrain), which is the only arm that can change the "
        "encoder."),
    "F4 artifact (nuisance control fires)": (
        "WITHHELD: the signed claim is withdrawn. The anatomically invariant nuisance control recovered the "
        "'axis', which a genuinely signed quantity cannot allow, so Lane A' reflects a magnitude/acquisition "
        "artifact, not a signed quantity."),
    "F5 not attributable (Ac not beaten)": (
        "WITHHELD (not attributable): Lane A' beats the binding bar and clears its null, but it does NOT "
        "beat the capacity-matched control Ac by the margin. The win rides on the head's generic capacity "
        "(nonlinearity, width, pair information), which Ac holds identical, rather than on the antisymmetry "
        "CONSTRAINT. Report as not attributable to antisymmetry rather than a constraint win."),
}
for _, r in futures_table.iterrows():
    print(f"### {r['future']}")
    print(f"  primary            : {r['primary']}")
    print(f"  mirror             : {r['mirror']}")
    print(f"  nuisance ok        : {r['nuisance_ok']}")
    print(f"  beats capacity ctrl: {r['beats_capacity_matched']}")
    print(f"  C-null significant : {r['c_null_significant']}  (floor characterization only)")
    print(f"  claim              : {CLAIMS[r['future']]}")
    print()

### F1 head beats bar (positive)
  primary            : ANTISYMMETRY BEATS BINDING BAR AND CAPACITY-MATCHED CONTROL
  mirror             : FLIPS (measured slope antisymmetric through the encoder)
  nuisance ok        : True
  beats capacity ctrl: True
  C-null significant : False  (floor characterization only)
  claim              : LICENSED: the antisymmetry-constrained head decodes the signed axis above the binding bar max(standard-ea59fea0, untrained-floor), beats the capacity-matched control Ac, AND clears its permutation null, and the MEASURED anatomical mirror flips it. Arm 1: the constraint extracts a signed axis the unconstrained probe missed. Arm 2: the equivariance loss built one. (The exact wiring -1 is a separate by-construction check, always true.)

### F2 decodable but non-flipping
  primary            : ANTISYMMETRY BEATS BINDING BAR AND CAPACITY-MATCHED CONTROL
  mirror             : DOES NOT FLIP (measured slope not antisymmetric)
  nuisance ok        : True
  beats ca

## 4. Expected-shape panels for each future

**Step 5 of 7. Every point in every panel is SIMULATED. These are not data.**

*What we are about to do.* For each of the five futures, synthesize a point cloud that reproduces its lane
A_prime R-squared and its measured anatomical-mirror slope, and draw two rows of panels: a decodability
scatter and a mirror scatter.

*Why draw pictures of results that do not exist.* Because a reader who has only seen numbers cannot
recognise an outcome on sight. Knowing what an R-squared of -0.10 looks like as a point cloud, versus
+0.42, is what allows the real figures in `nb_09a` to be read quickly and correctly. Note how the
simulator works: for a non-positive target R-squared it generates predictions INDEPENDENT of the target,
which is exactly the picture of a readout carrying no usable signal.

*One thing the mirror row is designed to teach.* Every panel draws the exact wiring-swap line `y = -x`.
It is true by construction for every future, so it is precisely NOT what distinguishes them. What
distinguishes them is the measured cloud. A reader who takes the presence of a `-1` line as evidence of
equivariance has misread the panel, and putting the line in every panel makes that impossible to sustain.

*What to look at in the output.* The saved path
`notes/ideas-claude/09-reflection-equivariant-symmetry-axis/images/idea9_possible_futures.png`, and a
`UserWarning` that the Agg backend cannot show figures interactively, which is expected and harmless. The
figure's own suptitle carries the disclosure: simulated expected shapes, NOT data.

*What we may NOT conclude.* Nothing. These are drawings of hypotheses.


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def simulate_scatter(r2, n=40, seed=0):
    '''Return (y_true, y_pred) with approximately the requested held-out R2.'''
    rng = np.random.default_rng(seed)
    y = rng.normal(0, 1, n)
    if r2 <= 0:
        pred = rng.normal(0, 1, n)
    else:
        noise_var = (1 - r2) / max(r2, 1e-6)
        pred = y + rng.normal(0, np.sqrt(noise_var), n)
    return y, pred


def simulate_mirror(slope, n=40, seed=0):
    rng = np.random.default_rng(seed)
    orig = rng.normal(0, 1, n)
    mir = slope * orig + rng.normal(0, 0.12, n)
    return orig, mir


ncol = len(FUTURES)
fig, axes = plt.subplots(2, ncol, figsize=(3.6 * ncol, 8))
for col, f in enumerate(FUTURES):
    seed = RANDOM_SEED + col
    yt, yp = simulate_scatter(f["a_r2"], seed=seed)
    ax = axes[0, col]
    ax.scatter(yt, yp, s=24, c="#e07a4b", edgecolors="#a44c26", linewidths=0.5)
    lim = 3.2
    ax.plot([-lim, lim], [-lim, lim], "--", color="#5f9e7e", linewidth=1.3)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    bar = max(f["d_r2"], f["c_r2"])
    ax.set_title(f"{f['title']}\nA' R2~{f['a_r2']:.2f}  bar max(D,C)~{bar:.2f}", fontsize=8)
    if col == 0:
        ax.set_ylabel("decoded signed scalar")
    ax.set_xlabel("ground-truth signed target")
    orig, mir = simulate_mirror(f["mirror_slope"], seed=seed + 100)
    ax2 = axes[1, col]
    mlim = 3.2
    xs = np.linspace(-mlim, mlim, 40)
    ax2.plot(xs, -xs, "-", color="#2f6f99", lw=1.6, label="wiring -1 (exact)")
    ax2.scatter(orig, mir, s=24, c="#e07a4b", edgecolors="#a44c26", linewidths=0.5, label="measured")
    ax2.axhline(0, color="#c4cdd8", lw=0.8); ax2.axvline(0, color="#c4cdd8", lw=0.8)
    ax2.set_xlim(-mlim, mlim); ax2.set_ylim(-mlim, mlim)
    flips = "FLIPS" if MIRROR_BAND[0] <= f["mirror_slope"] <= MIRROR_BAND[1] else "no flip"
    ax2.set_title(f"measured slope~{f['mirror_slope']:+.2f} ({flips})", fontsize=8)
    if col == 0:
        ax2.set_ylabel("decoded on mirrored input")
        ax2.legend(loc="upper right", fontsize=6)
    ax2.set_xlabel("decoded on original input")

fig.suptitle("Idea 9 possible futures (simulated expected shapes, NOT data). "
             "Blue line: exact wiring -1 (always true). Orange: measured anatomical mirror.", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.95])
OUT = IDEA9_DIR / "images"
try:
    OUT.mkdir(parents=True, exist_ok=True)
    out_path = OUT / "idea9_possible_futures.png"
    fig.savefig(out_path, dpi=120)
    print(f"saved {out_path}")
except Exception as exc:
    print(f"(figure not saved to images/: {exc})")
plt.show()

saved /Users/pmui/dev/alexpose/experiments/sjepa/gavd6-pm/notes/ideas-claude/09-reflection-equivariant-symmetry-axis/images/idea9_possible_futures.png


/var/folders/7t/kps5880d0gz_kzh7t6rvsw280000gn/T/ipykernel_6154/3233657247.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. External multi-view reach arm (non-clinical scaffold)

**Step 6 of 7. The real loaders are marked TODO and are NOT wired. The numbers this cell prints come from
a fixture whose answers were PLANTED by construction.**

*What we are about to do.* Define a loader interface for two public multi-view pose cohorts, define the
two reach metrics, and then exercise both metrics on a small synthetic multi-view fixture so the scaffold
is known to run.

*Why the reach arm is framed as non-clinical, and why there is no clinical transfer test.* The claim on
this cohort is internal-validity only: 18 canonical source videos, a transductive encoder, monocular
capture. A public cohort that is simultaneously clinical, skeleton-based, and participant-disjoint from
this one does not exist, so there is NO honest skeleton-level clinical transfer test available. What does
exist is a test of the reflection-equivariance PROPERTY itself, which is a geometric claim rather than a
clinical one.

*The two cohorts named.* CASIA-B (Yu, Tan, Tan 2006), 124 subjects across 11 camera views from 0 to 180
degrees, non-clinical gait. OU-MVLP-Pose (Takemura et al. 2018), about 10,000 subjects, multiple views,
pose keypoints released.

*The two reach questions, both symmetry-specific and view-specific.*

1. **View stability.** Is the signed axis decoded from one view consistent with the same subject decoded
   from a nearby view? If it is not, the axis is a property of the CAMERA rather than of the gait.
2. **Genuine mirror.** A true left-versus-right camera swap is a REAL physical reflection, not a synthetic
   x-negation. It should flip the signed axis. CASIA-B's symmetric view angles around 90 degrees provide a
   real-world analogue of the anatomical mirror, which would let the flip slope be measured on genuine
   reflections rather than on a transform we applied ourselves. That distinction is the entire value of
   this arm.

*Why real downloads are deliberately not wired.* These are large licensed datasets and this pass does not
fetch them. `load_casia_b` and `load_ou_mvlp_pose` return an empty list when called with no root, so the
scaffold degrades cleanly, and raise `NotImplementedError` if a root IS supplied, so nobody can mistake a
stub for a working parser. Both carry `TODO(real)` markers.

*What to look at in the output, and how to read it correctly. This is the most misreadable cell in the
notebook.*

- `external reach scaffold exercised on 18 synthetic multi-view clips (6 subjects x 3 views)`.
- `view-stability corr(axis@54deg, axis@126deg) = +1.000`.
- `genuine-mirror flip slope on fixture = -1.000`.

Those two numbers look like a spectacular external validation. **They are not any kind of validation, and
they are not external.** Both are PLANTED BY CONSTRUCTION:

- The fixture gives each subject a FIXED signed lean that is written into every view before the view
  rotation is applied, so the axis is identical across views up to the rotation. A correlation near +1 is
  therefore arithmetic, not evidence.
- The mirror slope is computed on the RAW coordinate axis, `signed_axis`, using the same six-pair
  negate-and-swap operator that defines the axis. `nb_09a` step 2 already proved that quantity is exactly
  antisymmetric on raw coordinates. So a slope of -1.000 here restates a proven identity. It says nothing
  about any encoder, because no encoder is involved in this cell at all.

*What the fixture genuinely establishes.* That the metric code runs, that the pivot and correlation
plumbing is correct, and that the interface is shaped to accept real clips. That is the whole of it.

*What it does NOT establish.* No external validity, no view invariance of the learned representation, no
measured equivariance of any encoder, and nothing clinical. This arm is reach-tier: it tests the
reflection property, not any diagnosis. The code's own closing note says exactly this and should be left
in place.


In [6]:
from dataclasses import dataclass


@dataclass
class MultiViewClip:
    subject_id: str
    view_deg: float
    coords: np.ndarray       # [T, 33, 3] normalized pose
    cohort: str = "synthetic"


def load_casia_b(root=None):
    '''TODO(real): parse a CASIA-B pose export at `root` into a MultiViewClip list.
    Not wired in this pass (large licensed dataset). Returns [] so the scaffold degrades cleanly.'''
    if root is None:
        return []
    raise NotImplementedError("Real CASIA-B loading is a marked TODO; provide a parser at the call site.")


def load_ou_mvlp_pose(root=None):
    '''TODO(real): parse OU-MVLP-Pose keypoints at `root`. Not wired in this pass.'''
    if root is None:
        return []
    raise NotImplementedError("Real OU-MVLP-Pose loading is a marked TODO.")


LEFT_RIGHT_PAIRS = [(11, 12), (23, 24), (25, 26), (27, 28), (29, 30), (31, 32)]


def signed_axis(coords):
    total = 0.0
    for li, ri in LEFT_RIGHT_PAIRS:
        total += coords[:, li, :].std(axis=0).sum() - coords[:, ri, :].std(axis=0).sum()
    return float(total)


def synthetic_multiview_fixture(n_subjects=6, views=(54.0, 90.0, 126.0), frames=40, seed=RANDOM_SEED):
    '''Tiny multi-view fixture: each subject has a fixed signed lean visible from every view; the view
    rotates the pose in the x-z plane so the signed x-excursion is genuinely view-dependent.'''
    rng = np.random.default_rng(seed)
    clips = []
    for s in range(n_subjects):
        lean = 1.0 if s % 2 == 0 else -1.0
        phase = np.linspace(0, 4 * np.pi, frames, endpoint=False)
        base = np.zeros((frames, 33, 3), dtype=np.float32)
        for j in range(33):
            base[:, j, :] = rng.normal(0, 0.04, 3)[None, :] + 0.03 * np.sin(phase + j)[:, None]
        for li, ri in LEFT_RIGHT_PAIRS:
            base[:, li, 0] += 0.05 * lean * np.sin(phase)
            base[:, ri, 0] -= 0.05 * lean * np.sin(phase)
        for v in views:
            theta = np.deg2rad(v - 90.0)
            rot = np.array([[np.cos(theta), 0, np.sin(theta)], [0, 1, 0], [-np.sin(theta), 0, np.cos(theta)]])
            coords = base @ rot.T
            clips.append(MultiViewClip(subject_id=f"subj{s:02d}", view_deg=float(v), coords=coords.astype(np.float32)))
    return clips


def genuine_mirror_view(coords):
    '''A real left/right reflection: negate the x-axis and swap the six L/R pairs (as a camera on the
    opposite side would see). Used to MEASURE the flip slope on real reflections when a cohort is wired.'''
    out = coords.copy()
    out[:, :, 0] = -out[:, :, 0]
    for li, ri in LEFT_RIGHT_PAIRS:
        out[:, [li, ri], :] = out[:, [ri, li], :]
    return out


clips = synthetic_multiview_fixture()
print(f"external reach scaffold exercised on {len(clips)} synthetic multi-view clips "
      f"({len({c.subject_id for c in clips})} subjects x {len({c.view_deg for c in clips})} views).")

df = pd.DataFrame([{"subject": c.subject_id, "view": c.view_deg, "axis": signed_axis(c.coords)} for c in clips])
wide = df.pivot(index="subject", columns="view", values="axis")
views_sorted = sorted(df["view"].unique())
stability = float("nan")
if len(views_sorted) >= 2:
    v_lo, v_hi = views_sorted[0], views_sorted[-1]
    stability = float(np.corrcoef(wide[v_lo], wide[v_hi])[0, 1])
    print(f"view-stability corr(axis@{v_lo:.0f}deg, axis@{v_hi:.0f}deg) = {stability:+.3f}  (expect strong positive)")

# genuine-mirror flip check on the fixture (measured, not by construction): axis(mirror(x)) vs axis(x)
frontal = [c for c in clips if c.view_deg == 90.0]
ax_o = np.array([signed_axis(c.coords) for c in frontal])
ax_m = np.array([signed_axis(genuine_mirror_view(c.coords)) for c in frontal])
mirror_slope = float(np.polyfit(ax_o, ax_m, 1)[0]) if len(ax_o) >= 2 else float("nan")
print(f"genuine-mirror flip slope on fixture = {mirror_slope:+.3f}  (expect near -1 for this raw axis)")
print("\nNOTE: synthetic fixture only. Real CASIA-B / OU-MVLP-Pose loading is a marked TODO. "
      "This arm is NON-CLINICAL and reach-tier; it tests the reflection property, not any diagnosis.")

external reach scaffold exercised on 18 synthetic multi-view clips (6 subjects x 3 views).
view-stability corr(axis@54deg, axis@126deg) = +1.000  (expect strong positive)
genuine-mirror flip slope on fixture = -1.000  (expect near -1 for this raw axis)

NOTE: synthetic fixture only. Real CASIA-B / OU-MVLP-Pose loading is a marked TODO. This arm is NON-CLINICAL and reach-tier; it tests the reflection property, not any diagnosis.


## 6. Persist the futures and decision bundle

**Step 7 of 7.**

*What we are about to do.* Write the futures table, the decision map, the gate constants, and the
reach-scaffold status to a JSON bundle next to the proposal folder.

*Why persist a simulation at all.* Three reasons. So the methodology document and README can cite the
exact simulated shapes rather than paraphrasing them. So a reader can DIFF the real
`idea9_antisymmetric_readout_result.json` from `nb_09a` against the canonical futures recorded here, which
is how the F3-versus-F4 miss in section 3 becomes checkable rather than a matter of memory. And so the
gate constants exist in machine-readable form, pinned to the moment before the result was known.

*The `supersedes` field, which should be kept and understood.* The bundle records that the
**proposal-level gates in the idea 9 README and METHODOLOGY worked example are SUPERSEDED by these
hardened gates**: the binding bar `max(D, C)`, the permutation nulls, the absolute nuisance control, and
the y-quality gate. Concretely, Idea 5's two proposal-level gates are gone: the "reach at least 80 percent
of the raw-coordinate null" gate, because lane B is near-circular and sits at 1.000 which makes that bar an
artifact of how the target was built, and the fixed "sign correct on at least 75 percent of held-out
sources" gate, for the sample-size reason in step 1. If a document quotes the 80 percent or 75 percent
gates as current, that document is out of date. See `IMPLEMENTATION.md`.

*What to look at in the output.* The written path
`notes/ideas-claude/09-reflection-equivariant-symmetry-axis/idea9_futures_bundle.json`, then the echoed
gate constants and `n_futures: 5`.

*One caveat about a field in the bundle.* `external_reach.fixture_view_stability_corr` and
`external_reach.fixture_genuine_mirror_slope` record the +1.000 and -1.000 from step 6. The field names
say `fixture` for a reason: as step 6 explained, both were planted by construction. Anyone reading this
bundle downstream must not lift those two values out as external validation.


In [7]:
import json
bundle = {
    "notebook": "nb_09c_futures_and_reach",
    "gates": {"floor_margin": FLOOR_MARGIN, "perm_alpha": PERM_ALPHA,
              "nuisance_abs": NUISANCE_ABS, "y_between_min": Y_BETWEEN_MIN,
              "mirror_band": list(MIRROR_BAND)},
    "futures": rows,
    "decision": {r["future"]: {"primary": r["primary"], "mirror": r["mirror"],
                               "nuisance_ok": bool(r["nuisance_ok"]),
                               "beats_capacity_matched": bool(r["beats_capacity_matched"]),
                               "c_null_significant": bool(r["c_null_significant"]),
                               "claim": CLAIMS[r["future"]]}
                 for r in rows},
    "external_reach": {
        "cohorts": ["CASIA-B (Yu 2006)", "OU-MVLP-Pose (Takemura 2018)"],
        "status": "scaffold only; real loaders are marked TODO",
        "tier": "non-clinical reach; tests reflection stability + genuine-mirror flip, not diagnosis",
        "fixture_view_stability_corr": stability,
        "fixture_genuine_mirror_slope": mirror_slope,
    },
    "supersedes": "The proposal-level gates in the idea-9 README/METHODOLOGY worked example are superseded "
                  "by these hardened gates (binding bar max(D,C), permutation nulls, absolute nuisance "
                  "control, y-quality gate). See IMPLEMENTATION.md.",
    "notes": "Simulated expected shapes, NOT data. The clinical claim on this cohort is internal-validity "
             "only. The exact wiring-swap slope -1 is a by-construction check (nb_09a section 5), not a "
             "future. Folder labels are dataset annotations, not diagnoses; all results transductive.",
}
out = IDEA9_DIR / "idea9_futures_bundle.json"
try:
    out.write_text(json.dumps(bundle, indent=2))
    print(f"wrote {out}")
except Exception as exc:
    print(f"(bundle not written: {exc})")
print(json.dumps({"gates": bundle["gates"], "n_futures": len(rows)}, indent=2))

wrote /Users/pmui/dev/alexpose/experiments/sjepa/gavd6-pm/notes/ideas-claude/09-reflection-equivariant-symmetry-axis/idea9_futures_bundle.json
{
  "gates": {
    "floor_margin": 0.05,
    "perm_alpha": 0.05,
    "nuisance_abs": 0.05,
    "y_between_min": 0.3,
    "mirror_band": [
      -1.25,
      -0.8
    ]
  },
  "n_futures": 5
}


## Summary: what notebook 09c established, and what it deliberately did not

**What this notebook is.** A pre-registration aid and a code scaffold. It contains NO empirical result.
Every lane value, p-value, variance fraction, and mirror slope in sections 2 through 4 was chosen by hand
to probe a decision boundary. Every number in section 5 comes from a synthetic fixture whose answers were
planted by construction.

**What it established.**

1. The hardened decision rule is total and self-consistent: five simulated futures, one shared scoring
   function, and an explicit pre-committed claim for each verdict, including what each verdict WITHHOLDS.
2. The two traps are real and were worth building lanes for: a strong-looking treatment is not enough on
   its own if a side-blind nuisance control also recovers the axis (F4), or if a capacity-matched control
   is not beaten (F5).
3. The reach-arm interface runs, with its metrics exercised end to end on a fixture.

**The rehearsal's genuine miss, kept visible on purpose.** Future **F3 predicted an INFORMATIVE NULL for
arm 1**. The **actual arm 1 verdict was `ARTIFACT (side-agnostic nuisance control fired)`**, which is this
notebook's F4. The direction was predicted correctly; the MECHANISM was not. Nobody anticipated that a
control which is mathematically blind to left and right would OUTSCORE the antisymmetric treatment, by
0.140 (lane E at -0.066 against A_prime at -0.206). The lesson is that a control ladder must be richer
than the set of outcomes you expect: include a control for every escape route you can NAME, not only for
every result you can PREDICT. Had lane E been omitted, arm 1 would have been published as the informative
null this notebook rehearsed, and that write-up would have been wrong. The prediction has NOT been edited
to match the result.

**One superseded input to be aware of.** F3's anchor values (A -0.187, C +0.147, D -0.014, mirror -0.34)
came from a superseded Idea 5 bundle carrying the `d0acc262` fingerprint. The authoritative Idea 5 values
on the `ea59fea0` checkpoint are A -0.602, C -0.156, D -0.131, and mirror slope -0.741. The code cell
retains the superseded inputs because they are what was actually scored; F3's verdict is unchanged under
the correction.

**What the reach arm does NOT show.** The fixture's view-stability correlation of +1.000 and genuine-mirror
slope of -1.000 were PLANTED by construction and are not external validity of anything. The real CASIA-B
and OU-MVLP-Pose loaders remain marked TODO and are not wired. This arm is non-clinical and reach-tier: it
tests the reflection property, never a diagnosis.

**What is superseded elsewhere by this notebook's gates.** The proposal-level gates in the idea 9 README
and METHODOLOGY worked example, specifically the "80 percent of the raw-coordinate null" and "75 percent
sign consistency" gates, are superseded by the hardened gates recorded in the bundle here.

**Where to go for actual results.** `nb_09a` for arm 1 on the frozen encoder, verdict
`ARTIFACT (side-agnostic nuisance control fired)`, which also measured the fact that binds the whole
family: only about 7.5 percent of the target's variance lies between the 18 source videos, against a
preregistered 30 percent. `new_nb_09_00` through `new_nb_09_03` for arm 2, verdict `NO CREDIT`.

**Standing caveats.** Folder labels are dataset annotations, not clinical diagnoses. All results on this
cohort are transductive and the source video is the independent unit of evidence. Nothing here is clinical
validation and nothing here is unseen-video performance.
